# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark RDD - SOLUTION
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful

In [26]:
from pyspark import SparkContext, SparkConf
import numpy as np
import operator

In [27]:
conf=SparkConf().setAppName("Lab4-rdd").setMaster("local[*]")
sc = SparkContext(conf=conf)

Using PySpark and RDD's on the https://coding.csel.io machines is slow -- most of the code is executed in Python and this is much less efficient than the java-based code using the PySpark dataframes. Be patient and trying using `.cache()` to cache the output of joins. You may want to start with a reduced set of data before running the full task. You can use the `sample()` method to extract just a sample of the data or use 

These two RDD's are called "rawCitations" and "rawPatents" because you probably want to process them futher (e.g. convert them to integer types, etc). 

The `textFile` function returns data in strings. This should work fine for this lab.

Other methods you use might return data in type `Byte`. If you haven't used Python `Byte` types before, google it. You can convert a value of `x` type byte into e.g. a UTF8 string using `x.decode('uft-8')`. Alternatively, you can use the `open` method of the gzip library to read in all the lines as UTF-8 strings like this:
```
import gzip
with gzip.open('cite75_99.txt.gz', 'rt',encoding='utf-8') as f:
    rddCitations = sc.parallelize( f.readlines() )
```
This is less efficient than using `textFile` because `textFile` would use the underlying HDFS or other file system to read the file across all the worker nodes while the using `gzip.open()...readlines()` will read all the data in the frontend and then distribute it to all the worker nodes.

In [28]:
rddCitations = sc.textFile("cite75_99.txt.gz")
rddPatents = sc.textFile("apat63_99.txt.gz")

The data looks like the following.

In [29]:
rddCitations.take(5)

['"CITING","CITED"',
 '3858241,956203',
 '3858241,1324234',
 '3858241,3398406',
 '3858241,3557384']

In [30]:
rddPatents.take(5)

['"PATENT","GYEAR","GDATE","APPYEAR","COUNTRY","POSTATE","ASSIGNEE","ASSCODE","CLAIMS","NCLASS","CAT","SUBCAT","CMADE","CRECEIVE","RATIOCIT","GENERAL","ORIGINAL","FWDAPLAG","BCKGTLAG","SELFCTUB","SELFCTLB","SECDUPBD","SECDLWBD"',
 '3070801,1963,1096,,"BE","",,1,,269,6,69,,1,,0,,,,,,,',
 '3070802,1963,1096,,"US","TX",,1,,2,6,63,,0,,,,,,,,,',
 '3070803,1963,1096,,"US","IL",,1,,2,6,63,,9,,0.3704,,,,,,,',
 '3070804,1963,1096,,"US","OH",,1,,2,6,63,,3,,0.6667,,,,,,,']

In other words, they are a single string with multiple CSV's. You will need to convert these to (K,V) pairs, probably convert the keys to `int` and so on. You'll need to `filter` out the header string as well since there's no easy way to extract all the lines except the first.

In [31]:
# Grab the header lines so we can filter them out later.
citationHeader = rddCitations.first()
patentHeader = rddPatents.first()
print(citationHeader)
print(patentHeader)

"CITING","CITED"
"PATENT","GYEAR","GDATE","APPYEAR","COUNTRY","POSTATE","ASSIGNEE","ASSCODE","CLAIMS","NCLASS","CAT","SUBCAT","CMADE","CRECEIVE","RATIOCIT","GENERAL","ORIGINAL","FWDAPLAG","BCKGTLAG","SELFCTUB","SELFCTLB","SECDUPBD","SECDLWBD"


In [32]:
SAMPLE = False   # set to True for a fast 5% development run

if SAMPLE:
    citations_rdd = rddCitations.sample(False, 0.05, seed=42)
    patents_rdd = rddPatents.sample(False, 0.05, seed=42)
else:
    citations_rdd = rddCitations
    patents_rdd = rddPatents

In [33]:
# Turn each citation line into (CITED, CITING) - CITED is the key so we can join on it.
citationPairs = (
    citations_rdd.filter(lambda line: line != citationHeader)
    .map(lambda line: line.split(","))
    .map(lambda fields: (int(fields[1]), int(fields[0])))
)
# (CITED, CITING)
citationPairs.take(5)

[(956203, 3858241),
 (1324234, 3858241),
 (3398406, 3858241),
 (3557384, 3858241),
 (3634889, 3858241)]

In [35]:
# Build the lookup table of (patent, state), skipping patents with no state.
# Field 0 is PATENT, field 5 is POSTATE.
def parsePatentState(line):
    fields = line.split(",")
    return (int(fields[0]), fields[5].strip('"'))

# (PATENT, STATE) for patents that actually have a state
patentStates = patents_rdd.filter(lambda line: line != patentHeader) \
                          .map(parsePatentState) \
                          .filter(lambda kv: kv[1] != "")

patentStates.cache()
patentStates.take(5)

[(3070802, 'TX'),
 (3070803, 'IL'),
 (3070804, 'OH'),
 (3070805, 'CA'),
 (3070806, 'PA')]

In [36]:
# Join 1: attach the cited patent's state.
citedJoined = citationPairs.join(patentStates)
citedJoined.cache()
# (CITED, (CITING, CITED_STATE))
citedJoined.take(5)

[(3726800, (3859044, 'NY')),
 (3726800, (4195124, 'NY')),
 (3726800, (4377489, 'NY')),
 (3726800, (4485028, 'NY')),
 (3726800, (5180514, 'NY'))]

In [37]:
# Re-key by CITING, then perform Join 2 to attach the citing patent's state.
bothJoined = citedJoined.map(lambda x: (x[1][0], x[1][1])) \
                        .join(patentStates)

bothJoined.cache()
# (CITING, (CITED_STATE, CITING_STATE))
bothJoined.take(5)

[(5906043, ('CO', 'CA')),
 (5906043, ('CA', 'CA')),
 (5906043, ('PA', 'CA')),
 (5906043, ('NY', 'CA')),
 (5906043, ('OR', 'CA'))]

In [38]:
# Keep only same-state pairs, then add up a 1 for each one per patent.
sameStateCounts = (
    bothJoined.filter(lambda x: x[1][0] == x[1][1]).
    map(lambda x: (x[0], 1))
    .reduceByKey(operator.add)
)

sameStateCounts.cache()
sameStateCounts.take(5)

[(5906043, 74), (4373847, 5), (5733390, 1), (5546628, 1), (4627419, 3)]

In [40]:
# Attach the count to the end of each full patent record.
# (PATENT, [all other fields as strings])
patentFields = (
    patents_rdd.filter(lambda line: line != patentHeader)
    .map(lambda line: line.split(","))
    .map(lambda fields: (int(fields[0]), fields[1:]))
)

# (PATENT, [other fields ..., SAME_STATE])
newPatents = (
    patentFields.join(sameStateCounts)
    .map(lambda x: (x[0], x[1][0] + [x[1][1]]))
)

newPatents.take(2)

[(5958744,
  ['1999',
   '14515',
   '1998',
   '"US"',
   '"MI"',
   '758832',
   '2',
   '',
   '435',
   '3',
   '33',
   '4',
   '0',
   '1',
   '',
   '0',
   '',
   '6',
   '0',
   '0',
   '',
   '',
   2]),
 (3885612,
  ['1975',
   '5625',
   '1973',
   '"US"',
   '"NV"',
   '473130',
   '2',
   '1',
   '144',
   '5',
   '51',
   '6',
   '3',
   '1',
   '0.6667',
   '0.5',
   '15',
   '3.6667',
   '0',
   '0',
   '0',
   '0',
   3])]

In [41]:
# Get the 10 patents with the highest count.
top10 = newPatents.takeOrdered(10, key=lambda x: (-x[1][-1], x[0]))

# RDD form, as in top-subsample-rdd.png
for record in top10:
    print(record)

# compact form, directly comparable with the DataFrame notebook's output
print()
print("PATENT     POSTATE  SAME_STATE")
for patent, fields in top10:
    print("%-10d %-8s %d" % (patent, fields[4].strip('"'), fields[-1]))

(5959466, ['1999', '14515', '1997', '"US"', '"CA"', '5310', '2', '', '326', '4', '46', '159', '0', '1', '', '0.6186', '', '4.8868', '0.0455', '0.044', '', '', 125])
(5983822, ['1999', '14564', '1998', '"US"', '"TX"', '569900', '2', '', '114', '5', '55', '200', '0', '0.995', '', '0.7201', '', '12.45', '0', '0', '', '', 103])
(6008204, ['1999', '14606', '1998', '"US"', '"CA"', '749584', '2', '', '514', '3', '31', '121', '0', '1', '', '0.7415', '', '5', '0.0085', '0.0083', '', '', 100])
(5952345, ['1999', '14501', '1997', '"US"', '"CA"', '749584', '2', '', '514', '3', '31', '118', '0', '1', '', '0.7442', '', '5.1102', '0', '0', '', '', 98])
(5958954, ['1999', '14515', '1997', '"US"', '"CA"', '749584', '2', '', '514', '3', '31', '116', '0', '1', '', '0.7397', '', '5.181', '0', '0', '', '', 96])
(5998655, ['1999', '14585', '1998', '"US"', '"CA"', '', '1', '', '560', '1', '14', '114', '0', '1', '', '0.7387', '', '5.1667', '', '', '', '', 96])
(5936426, ['1999', '14466', '1997', '"US"', '"CA"

In [25]:
# Shut down Spark.
sc.stop()